# Dezvoltare software analiza date
## Tema 1 - Diferențe regionale în Uniunea Europeană

În această temă lucrați cu date reale **Eurostat** la nivel de regiune **NUTS2** (244 de regiuni din cele 27 de state membre). Folosiți ce am făcut la seminarii: selecție și filtrare, valori lipsă, `merge`/`join`, `groupby` + `agg`/`transform`, medii ponderate (seminarul 3), calcul matriceal cu `numpy` (seminarul 4) și grafice `matplotlib`.

**Anul de referință: 2023.**

### Datele (`data_in/`)

`Indicatori.csv` conține câte o linie pentru fiecare regiune NUTS2 (`Cod`, de exemplu `RO32`) și următorii indicatori, pentru anul 2023:

| Coloană | Indicator | Unitate |
|---|---|---|
| `PIB` | PIB pe locuitor | PPS (standard al puterii de cumpărare) |
| `Somaj` | Rata șomajului, 15-74 ani | % |
| `SperantaViata` | Speranța de viață la naștere | ani |
| `Tertiar` | Populația 25-64 ani cu studii superioare | % |
| `Populatie` | Populația la 1 ianuarie | persoane |
| `CD` | Cheltuieli de cercetare-dezvoltare | % din PIB |

Unele valori lipsesc (celule goale).

| Fișier | Coloane |
|---|---|
| `NUTS2.csv` | `Cod`, `Denumire` (denumirea regiunii) |
| `Tari.csv` | `CodTara`, `Tara`, `AnAderare`, `Euro` (1 = zona euro) |

Codul de țară este format din primele două litere ale codului regiunii (`RO32` → `RO`). Grecia are codul `EL`.

*Sursa datelor: Eurostat*

### Ce predați
- notebook-ul rulat complet (`Restart & Run All`), fără erori;
- folderul `data_out/` cu fișierele `Cerinta1.csv` ... `Cerinta9.csv` și cele 3 imagini de la cerința 10;
- după fiecare cerință, celula **Interpretare** completată cu 2-3 propoziții **scrise de voi**: ce arată rezultatul, nu ce face codul.

### Punctaj
Cerințele 1-9: câte 8 puncte (5 pentru cod și fișier, 3 pentru interpretare). Cerința 10: 28 de puncte (3 × 6 pentru grafice, 10 pentru concluzie). Total: 100 de puncte.

*Atenție:* cerințele se bazează una pe alta. Dacă vă blocați la una, salvați un rezultat parțial și mergeți mai departe.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from scipy.spatial.distance import cdist

Path("./data_out").mkdir(exist_ok=True)

## Cerința 1 - Citirea și combinarea datelor

Citiți cele trei fișiere și construiți un singur tabel cu câte o linie pentru fiecare regiune (index `Cod`), care să conțină indicatorii, denumirea regiunii, codul țării și informațiile despre țară. Salvați tabelul în *Cerinta1.csv*.

Afișați apoi numărul de valori lipsă pentru fiecare indicator și regiunile care au cel puțin o valoare lipsă.

**Verificare:** tabelul are 244 de linii, iar fiecare regiune are o denumire și o țară.

In [2]:

import os

df_ind = pd.read_csv("./data_in/Indicatori.csv")
df_nuts = pd.read_csv("./data_in/NUTS2.csv")
df_tari = pd.read_csv("./data_in/Tari.csv")

df_ind["CodTara"] = df_ind["Cod"].str[:2]

df_merged = pd.merge(df_ind, df_nuts, on="Cod", how="inner")
df_c1 = pd.merge(df_merged, df_tari, on="CodTara", how="left") #c1 = cerinta1
df_c1.set_index("Cod")

Path("./data_out").mkdir(exist_ok=True)
df_c1.to_csv("./data_out/Cerinta1.csv")

print(f"Număr total de linii (regiuni): {len(df_c1)}")

indicatori = ["PIB", "Somaj", "SperantaViata", "Tertiar", "Populatie", "CD"]
valori_lipsa_per_indicator = df_c1[indicatori].isna().sum()
print("\nNumărul de valori lipsă pentru fiecare indicator:")
print(valori_lipsa_per_indicator)

regiuni_cu_nan = df_c1[df_c1[indicatori].isna().any(axis=1)]
print(f"\nNumăr de regiuni cu cel puțin o valoare lipsă: {len(regiuni_cu_nan)}")
print("Regiunile cu valori lipsă:")
print(regiuni_cu_nan[["Denumire", "Tara"] + indicatori])


Număr total de linii (regiuni): 244

Numărul de valori lipsă pentru fiecare indicator:
PIB               0
Somaj             5
SperantaViata     0
Tertiar           1
Populatie         0
CD               17
dtype: int64

Număr de regiuni cu cel puțin o valoare lipsă: 21
Regiunile cu valori lipsă:
           Denumire      Tara    PIB  Somaj  SperantaViata  Tertiar  \
35     Niederbayern  Germania  44500    NaN           80.9     26.0   
59            Trier  Germania  34400    NaN           80.9     29.2   
126      Guadeloupe    Franta  25300   18.6           80.3     27.5   
127      Martinique    Franta  26200   10.8           81.6     29.2   
128          Guyane    Franta  16300   14.0           79.4     23.7   
129      La Réunion    Franta  24600   19.0           82.1     24.4   
130         Mayotte    Franta  11000    NaN           75.0      NaN   
170       Groningen    Olanda  46100    4.2           80.7     43.9   
171  Friesland (NL)    Olanda  36900    4.0           81.8     

**Interpretare:**
Tabelul final conține toate cele 244 de regiuni din UE alături de nume și țara. Există câteva valori lipsă în setul de date, mai ales la cheltuielile pentru cercetare-dezvoltare și la șomaj.

## Cerința 2 - Tratarea valorilor lipsă

Înlocuiți fiecare valoare lipsă cu **media indicatorului în țara respectivă**. Dacă indicatorul lipsește pentru toată țara, folosiți media pe UE. Salvați rezultatul în *Cerinta2.csv*.

**Verificare:** după completare nu mai există valori lipsă.

*Întrebare:* de ce este mai potrivită media țării decât media UE? Pentru care țară nu a fost posibil acest lucru?

In [3]:
# from pathlib import Path
# print("Working directory:", Path.cwd())
# print("Output directory:", Path("./data_out").resolve())
# print("CSV exists:", Path("./data_out/Cerinta1.csv").exists())
#Nu imi afisa in File Explorer fisierul Cerinta1.csv

#Rezolvare ex 2

df_c2 = df_c1.copy()
indicatori = ["PIB", "Somaj", "SperantaViata", "Tertiar", "Populatie", "CD"]

#media pe tara
for ind in indicatori:
    df_c2[ind] = df_c2[ind].fillna(df_c2.groupby("CodTara")[ind].transform("mean"))


tari_fara_medie = {}
for ind in indicatori:
    tari_lipsa = df_c2[df_c2[ind].isna()]["Tara"].unique()
    if len(tari_lipsa) > 0:
        tari_fara_medie[ind] = list(tari_lipsa)

print("Țările pentru care s-a aplicat media UE (lipsă totală la nivel de țară):")
print(tari_fara_medie)

#media UE
for ind in indicatori:
    df_c2[ind] = df_c2[ind].fillna(df_c2[ind].mean())

valori_lipsa_ramase = df_c2[indicatori].isna().sum().sum()
print(f"Număr total de valori lipsă rămase: {valori_lipsa_ramase}")

df_c2.to_csv("./data_out/Cerinta2.csv")


Țările pentru care s-a aplicat media UE (lipsă totală la nivel de țară):
{'CD': ['Olanda']}
Număr total de valori lipsă rămase: 0


**Interpretare:**
Am preferat media pe țară în locul mediei UE deoarece regiunile au condiții economice și sociale similare, deci o regiune seamănă mult cu vecinele ei. Singurul caz în care nu s-a putut calcula media pe țară a fost pentru Olanda

## Cerința 3 - Profilul fiecărei țări

Pentru fiecare țară calculați: numărul de regiuni, PIB-ul mediu, minim și maxim, precum și **denumirea** (nu codul) regiunii celei mai bogate și a celei mai sărace. Salvați rezultatul în *Cerinta3.csv* (coloanele `NrRegiuni`, `PIB_mediu`, `PIB_min`, `PIB_max`, `RegiuneBogata`, `RegiuneSaraca`).

In [4]:
df_c3_i = df_c2.copy()

def calc_profil_tara(group):
    nr_regiuni = len(group)
    pib_mediu = group["PIB"].mean()
    pib_min = group["PIB"].min()
    pib_max = group["PIB"].max()
    regiune_saraca = group.loc[group["PIB"].idxmin(), "Denumire"]
    regiune_bogata = group.loc[group["PIB"].idxmax(), "Denumire"]

    return pd.Series(
        {
            "NrRegiuni": nr_regiuni,
            "PIB_mediu": pib_mediu,
            "PIB_min": pib_min,
            "PIB_max": pib_max,
            "RegiuneBogata": regiune_bogata,
            "RegiuneSaraca": regiune_saraca,
        }
    )

df_c3 = (df_c3_i.groupby("Tara", as_index=True).apply(calc_profil_tara, include_groups=False).reset_index())
df_c3.set_index("Tara")
df_c3.to_csv("./data_out/Cerinta3.csv")

print(df_c3.head())

       Tara  NrRegiuni     PIB_mediu  PIB_min  PIB_max  \
0   Austria          9  45844.444444    33200    56200   
1    Belgia         11  43818.181818    27800    73100   
2  Bulgaria          6  21183.333333    15700    38800   
3     Cehia          8  34612.500000    23800    75700   
4     Cipru          1  37400.000000    37400    37400   

                                       RegiuneBogata      RegiuneSaraca  
0                                           Salzburg         Burgenland  
1  Région de Bruxelles-Capitale/Brussels Hoofdste...      Prov. Hainaut  
2                                        Yugozapaden  Yuzhen tsentralen  
3                                              Praha        Severozápad  
4                                             Kýpros             Kýpros  


**Interpretare:**
În majoritatea țărilor cu mai multe regiuni, cea mai bogată este chiar regiunea capitalei sau un alt centru urban, în timp ce regiunile cu PIB minim sunt de obicei zone de provincie sau de graniță. Cipru are o singura zona deci ea reprezinta si minim si maxim.

## Cerința 4 - Diferențe în interiorul țărilor

Pentru țările cu **cel puțin 3 regiuni**, calculați:
- coeficientul de variație al PIB-ului regional: $CV = \dfrac{\sigma}{\bar{x}}$ (cu `ddof=0`);
- raportul dintre PIB-ul celei mai bogate și al celei mai sărace regiuni: $\dfrac{\max}{\min}$.

Sortați descrescător după $CV$ și salvați în *Cerinta4.csv* (coloanele `NrRegiuni`, `CV`, `MaxMin`).

*Întrebare:* pe ce loc este România? Ce regiune explică acest rezultat? Ce s-ar întâmpla cu $CV$ dacă ați elimina regiunea capitalei?

In [5]:
df_c4_i = df_c2.copy()

tari_3_plus = df_c4_i.groupby("Tara").filter(lambda g: len(g) >= 3)

def calc_diferente(g):
    nr_regiuni = len(g)
    pib_mean = g["PIB"].mean()
    pib_std = g["PIB"].std(ddof=0)
    cv = pib_std / pib_mean
    max_min = g["PIB"].max() / g["PIB"].min()

    return pd.Series({"NrRegiuni": int(nr_regiuni), "CV": cv, "MaxMin": max_min})

df_c4 = tari_3_plus.groupby("Tara").apply(calc_diferente, include_groups=False)
df_c4["NrRegiuni"] = df_c4["NrRegiuni"].astype(int)
df_c4 = df_c4.sort_values(by="CV", ascending=False)
df_c4.to_csv("./data_out/Cerinta4.csv")

print(df_c4)
#ROmania
#initial
loc_ro = df_c4.index.get_loc("Romania") + 1
print(f"România se află pe locul: {loc_ro}")

ro_reg = df_c4_i[df_c4_i["Tara"].str.contains("Romania", case=False)]
print("\nPIB-ul regiunilor din România:")
print(ro_reg[["Denumire", "PIB"]])

ro_fara_capit = ro_reg[~ro_reg["Denumire"].str.contains("Bucureşti", case=False)]
cv_initial = ro_reg["PIB"].std(ddof=0) / ro_reg["PIB"].mean()
cv_fara_capit = ro_fara_capit["PIB"].std(ddof=0) / ro_fara_capit["PIB"].mean()

print(f"\nCV inițial România: {cv_initial:.4f}")
print(f"CV fără regiunea Capitalei: {cv_fara_capit:.4f}")


            NrRegiuni        CV    MaxMin
Tara                                     
Romania             8  0.527314  3.877095
Ungaria             8  0.516254  3.442623
Slovacia            4  0.464401  2.775120
Cehia               8  0.454709  3.180672
Bulgaria            6  0.380764  2.471338
Croatia             4  0.372282  2.485149
Irlanda             3  0.331826  2.407500
Polonia            17  0.319726  2.921182
Belgia             11  0.315555  2.629496
Italia             21  0.285788  2.900000
Franta             27  0.264511  5.590909
Portugalia          9  0.251333  2.334928
Danemarca           5  0.243752  1.903790
Grecia             13  0.236661  2.265432
Germania           38  0.216530  2.373418
Spania             19  0.204149  2.082609
Olanda             12  0.190299  1.856742
Suedia              8  0.184374  1.773006
Finlanda            5  0.163070  1.495549
Austria             9  0.147699  1.692771
România se află pe locul: 1

PIB-ul regiunilor din România:
             Den

**Interpretare:**
Rezultatele ne arată că România are cele mai mari diferente economice interne din toată Uniunea Europeană. Motivul principal pentru ocuparea locului 1 este regiunea București-Ilfov, care este un centru economic uriaș in contrast cu restul țării. Fără regiunea capitalei, coeficientul de variație CV ar fi mult mai mic, arătând că între celelalte regiuni ale Tarii nu există diferențe atât de mari.

## Cerința 5 - Media simplă vs. media ponderată

Pentru fiecare țară calculați rata șomajului ca:
- medie **simplă** a regiunilor;
- medie **ponderată cu populația** regiunilor:

$$\bar{s}_{t} = \frac{\sum_{r \in t} s_r \cdot P_r}{\sum_{r \in t} P_r}$$

și diferența dintre ele (`SomajPonderat - SomajSimplu`). Sortați după valoarea absolută a diferenței, descrescător, și salvați în *Cerinta5.csv*.

*Întrebare:* în ce țări diferența este mare și cum o explicați? De ce diferența este exact 0 în unele țări?

In [6]:
df_c5_i = df_c2.copy()

def calc_somaj_tara(g):
    somaj_simplu = g["Somaj"].mean()
    somaj_ponderat = (g["Somaj"] * g["Populatie"]).sum() / g["Populatie"].sum()
    diferenta = somaj_ponderat - somaj_simplu
    dif_abs = abs(diferenta)

    return pd.Series(
        {
            "SomajSimplu": somaj_simplu,
            "SomajPonderat": somaj_ponderat,
            "Diferenta": diferenta,
            "DifAbs": dif_abs,
        }
    )
df_c5 = df_c5_i.groupby("Tara").apply(calc_somaj_tara, include_groups=False)
df_c5 = df_c5.sort_values(by="DifAbs", ascending=False)

df_c5_final = df_c5[["SomajSimplu", "SomajPonderat", "Diferenta"]]
df_c5_final.to_csv("./data_out/Cerinta5.csv")

print(df_c5_final.head())
print("\nȚări cu diferență mică / zero:")
print(df_c5_final.tail())

          SomajSimplu  SomajPonderat  Diferenta
Tara                                           
Spania      13.263158      12.252549  -1.010609
Franta       8.311538       7.403790  -0.907749
Austria      4.544444       5.139297   0.594852
Lituania     6.350000       6.938586   0.588586
Italia       7.600000       8.170133   0.570133

Țări cu diferență mică / zero:
           SomajSimplu  SomajPonderat     Diferenta
Tara                                               
Estonia            6.4            6.4 -8.881784e-16
Cipru              5.8            5.8  0.000000e+00
Malta              3.5            3.5  0.000000e+00
Letonia            6.5            6.5  0.000000e+00
Luxemburg          5.2            5.2  0.000000e+00


**Interpretare:**
Diferența apare în statele mari unde populația nu este împărțită egal între regiuni. Dacă o regiune aglomerată are un șomaj mic sau mare, ea trage după sine media reală a țării, detaliu pe care media simplă îl ignoră, deoarece nu are pondere. In cazul tarilor mici, formate dintr-o singură regiune (precum Cipru), diferența este 0 deoarece cele 2 medii reprezinta acelasi lucru.

## Cerința 6 - Politica de coeziune: regiunile mai puțin dezvoltate

Uniunea Europeană consideră **mai puțin dezvoltate** regiunile cu PIB pe locuitor sub **75%** din media UE.

1. Calculați media PIB pe locuitor la nivelul UE, **ponderată cu populația** (de ce nu media simplă a regiunilor?).
2. Adăugați coloanele `IndicePIB` $= \dfrac{PIB_r}{\overline{PIB}_{UE}} \cdot 100$ și `MaiPutinDezvoltata` (`True` dacă indicele este sub 75).
3. Pentru fiecare țară: numărul de regiuni, numărul de regiuni sub 75 și **ponderea populației** care trăiește în aceste regiuni.

Sortați descrescător după ponderea populației și salvați în *Cerinta6.csv* (coloanele `NrRegiuni`, `NrSub75`, `PondPopSub75`).

*Întrebare:* care regiuni din România sunt peste prag?

In [8]:
df_c6_i = df_c2.copy()
pib_mediu_ue = (df_c6_i["PIB"] * df_c6_i["Populatie"]).sum() / df_c6_i["Populatie"].sum()
print(f"Media PIB pe locuitor la nivel UE (ponderată): {pib_mediu_ue:.2f} PPS")
df_c6_i["IndicePIB"] = (df_c6_i["PIB"] / pib_mediu_ue) * 100
df_c6_i["MaiPutinDezvoltata"] = df_c6_i["IndicePIB"] < 75

def calc_coeziune(g):
    nr_regiuni = len(g)
    nr_sub75 = g["MaiPutinDezvoltata"].sum()
    pop_totala_tara = g["Populatie"].sum()
    pop_sub75 = g[g["MaiPutinDezvoltata"]]["Populatie"].sum()
    pond_pop_sub75 = (pop_sub75 / pop_totala_tara) * 100
    return pd.Series(
        {
            "NrRegiuni": int(nr_regiuni),
            "NrSub75": int(nr_sub75),
            "PondPopSub75": pond_pop_sub75,
        }
    )
df_c6 = df_c6_i.groupby("Tara").apply(calc_coeziune, include_groups=False)

df_c6["NrRegiuni"] = df_c6["NrRegiuni"].astype(int)
df_c6["NrSub75"] = df_c6["NrSub75"].astype(int)
df_c6 = df_c6.sort_values(by="PondPopSub75", ascending=False)

df_c6.to_csv("./data_out/Cerinta6.csv")
print(df_c6.head(10))


ro_coeziune = df_c6_i[df_c6_i["Tara"].str.contains("Romania", case=False)]
ro_peste_75 = ro_coeziune[~ro_coeziune["MaiPutinDezvoltata"]]

print("Regiunile din România cu PIB peste pragul de 75% din media UE:")
print(ro_peste_75[["Denumire", "PIB", "IndicePIB"]])

Media PIB pe locuitor la nivel UE (ponderată): 38354.60 PPS
            NrRegiuni  NrSub75  PondPopSub75
Tara                                        
Letonia             1        1    100.000000
Slovacia            4        3     86.583203
Ungaria             8        7     82.593244
Croatia             4        3     80.040375
Romania             8        6     79.211836
Lituania            2        1     70.296075
Bulgaria            6        5     68.724493
Portugalia          9        5     68.591005
Grecia             13       12     63.572841
Polonia            17       13     62.526422
Regiunile din România cu PIB peste pragul de 75% din media UE:
            Denumire    PIB   IndicePIB
222  Bucureşti-Ilfov  69400  180.943118
224             Vest  29400   76.653136


**Interpretare:**
Media ponderată la nivelul UE reflectă mai bine realitatea, luând în calcul numărul de locuitori din fiecare zonă. Pentru România, rezultatele arată că București-Ilfov și Vest trec peste pragul de dezvoltare de 75%.

## Cerința 7 - NumPy: matricea de corelație calculată de mână

Folosiți variabilele `PIB`, `Somaj`, `SperantaViata`, `Tertiar`, `CD` (fără `Populatie`), lucrând direct cu tablouri `numpy`.

1. Standardizați datele: $Z = \dfrac{X - \bar{x}}{\sigma}$ (`ddof=0`).
2. Calculați $R = \dfrac{1}{n} Z^T Z$.
3. Găsiți perechea de variabile **diferite** cu cea mai puternică corelație, în valoare absolută.

**Verificare:** $R$ coincide cu `np.corrcoef`.

Salvați $R$ (cu numele variabilelor pe linii și coloane) în *Cerinta7.csv*.

In [10]:
variabile_c7 = ["PIB", "Somaj", "SperantaViata", "Tertiar", "CD"]
X = df_c2[variabile_c7].values
mean = np.mean(X, axis=0)
std = np.std(X, axis=0, ddof=0)
Z = (X - mean) / std
n = Z.shape[0]
R = (1 / n) * np.dot(Z.T, Z)

R_corrcoef = np.corrcoef(X, rowvar=False)
verificare = np.allclose(R, R_corrcoef)
print(f"Verificare: R coincide cu np.corrcoef? {verificare}")
R_df = pd.DataFrame(R, index=variabile_c7, columns=variabile_c7)
R_no_diag_mat = R_df.to_numpy(copy=True)
np.fill_diagonal(R_no_diag_mat, 0)
R_no_diag = pd.DataFrame(R_no_diag_mat, index=variabile_c7, columns=variabile_c7)

abs_R = R_no_diag.abs()
var1 = abs_R.max().idxmax()
var2 = abs_R[var1].idxmax()
val_corr = R_df.loc[var1, var2]

print(f"Cea mai puternică corelație este între '{var1}' și '{var2}': {val_corr:.4f}")

R_df.to_csv("./data_out/Cerinta7.csv")

Verificare: R coincide cu np.corrcoef? True
Cea mai puternică corelație este între 'PIB' și 'Tertiar': 0.5826


**Interpretare:**
Rezultatul ne arată că cea mai puternică corelație pozitivă există între PIB și populația cu studii superioare. Ne indică o legătură directă între puterea economică a unei regiuni și gradul de educație al forței de muncă.

## Cerința 8 - Relațiile dintre indicatori diferă între statele vechi și cele noi?

Împărțiți regiunile în două grupuri: țări care au aderat **înainte de 2004** și țări care au aderat **din 2004 încoace**. Pentru fiecare grup, calculați de mână matricea de corelație (standardizare în interiorul grupului, apoi $R = \frac{1}{n} Z^T Z$), apoi diferența $D = R_{vechi} - R_{noi}$.

Salvați $D$ în *Cerinta8.csv* și identificați perechea de variabile pentru care diferența este cea mai mare, în valoare absolută.

**Verificare:** fiecare matrice coincide cu `np.corrcoef` aplicat pe grupul respectiv.

*Întrebări:* în ce grup sunt corelațiile, în general, mai puternice? Cum explicați semnul diferit al corelației dintre `Somaj` și `SperantaViata` în cele două grupuri?


In [11]:
df_c8_i = df_c2.copy()
variabile_c8 = ["PIB", "Somaj", "SperantaViata", "Tertiar", "CD"]

grup_vechi = df_c8_i[df_c8_i["AnAderare"] < 2004]
grup_noi = df_c8_i[df_c8_i["AnAderare"] >= 2004]

def calc_R_manual(df, cols):
    X = df[cols].values
    mean = np.mean(X, axis=0)
    std = np.std(X, axis=0, ddof=0)
    Z = (X - mean) / std
    n = Z.shape[0]
    R = (1 / n) * np.dot(Z.T, Z)
    return R, X

R_vechi, X_vechi = calc_R_manual(grup_vechi, variabile_c8)
R_noi, X_noi = calc_R_manual(grup_noi, variabile_c8)
verif_vechi = np.allclose(R_vechi, np.corrcoef(X_vechi, rowvar=False))
verif_noi = np.allclose(R_noi, np.corrcoef(X_noi, rowvar=False))
print(f"Verificare R_vechi coincide cu np.corrcoef? {verif_vechi}")
print(f"Verificare R_noi coincide cu np.corrcoef? {verif_noi}")

D = R_vechi - R_noi
D_df = pd.DataFrame(D, index=variabile_c8, columns=variabile_c8)

D_df.to_csv("./data_out/Cerinta8.csv")

D_no_diag_mat = D_df.to_numpy(copy=True)
np.fill_diagonal(D_no_diag_mat, 0)
D_no_diag = pd.DataFrame(D_no_diag_mat, index=variabile_c8, columns=variabile_c8)
abs_D = D_no_diag.abs()
var1_d = abs_D.max().idxmax()
var2_d = abs_D[var1_d].idxmax()
val_diff = D_df.loc[var1_d, var2_d]
print(f"\nCea mai mare diferență (în valoare absolută) este între '{var1_d}' și '{var2_d}': {val_diff:.4f}")
r_vechi_df = pd.DataFrame(R_vechi, index=variabile_c8, columns=variabile_c8)
r_noi_df = pd.DataFrame(R_noi, index=variabile_c8, columns=variabile_c8)

print(f"Corelație Șomaj - SperanțaViata (Țări Vechi): {r_vechi_df.loc['Somaj', 'SperantaViata']:.4f}")
print(f"Corelație Șomaj - SperanțaViata (Țări Noi): {r_noi_df.loc['Somaj', 'SperantaViata']:.4f}")

Verificare R_vechi coincide cu np.corrcoef? True
Verificare R_noi coincide cu np.corrcoef? True

Cea mai mare diferență (în valoare absolută) este între 'SperantaViata' și 'CD': -0.5945
Corelație Șomaj - SperanțaViata (Țări Vechi): 0.0958
Corelație Șomaj - SperanțaViata (Țări Noi): -0.4125


**Interpretare:**
Relația dintre cercetare și speranța de viață arată cea mai mare diferență între cele două grupuri. În țările aderate dupa 2004, șomajul scade speranța de viață, în timp ce în țările vechi șomerii sunt protejați de un sistem medical și social bine pus la punct.

## Cerința 9 - NumPy: regiunile cele mai asemănătoare cu regiunile României

1. Calculați matricea distanțelor euclidiene dintre toate regiunile, pe datele standardizate $Z$ de la cerința 7:
$$d_{rq} = \sqrt{\sum_{j} (z_{rj} - z_{qj})^2}$$
2. Pentru fiecare regiune din România (`RO11` ... `RO42`), găsiți cea mai apropiată regiune **dintr-o altă țară**.

**Verificare:** matricea coincide cu `scipy.spatial.distance.cdist`.

Salvați în *Cerinta9.csv* (index: codul regiunii din România; coloanele `Regiune`, `CelMaiApropiat`, `Denumire`, `Tara`, `Distanta`).

*Întrebare:* sunt rezultatele plauzibile? De ce distanța pentru București-Ilfov este mult mai mare decât pentru celelalte regiuni?

In [13]:
variabile_c9 = ["PIB", "Somaj", "SperantaViata", "Tertiar", "CD"]
X = df_c2[variabile_c9].values
mean = np.mean(X, axis=0)
std = np.std(X, axis=0, ddof=0)
Z = (X - mean) / std
dist_matrix = np.sqrt(np.sum((Z[:, np.newaxis, :] - Z[np.newaxis, :, :]) ** 2, axis=2))

dist_cdist = cdist(Z, Z, metric="euclidean")
verif_dist = np.allclose(dist_matrix, dist_cdist)
print(f"Verificare: matricea distanțelor coincide cu cdist? {verif_dist}")

df_c9_base = df_c2.copy()
coduri_ro = df_c9_base[df_c9_base["Tara"].str.contains("Romania", case=False)].index
rezultate_ro = []

for cod_ro in coduri_ro:
    idx_ro = df_c9_base.index.get_loc(cod_ro)
    nume_ro = df_c9_base.loc[cod_ro, "Denumire"]
    masca_alte_tari = ~df_c9_base["Tara"].str.contains("Romania", case=False).values
    distante_ro = dist_matrix[idx_ro]
    distante_alte_tari = np.where(masca_alte_tari, distante_ro, np.inf)
    idx_min = np.argmin(distante_alte_tari)
    cod_apropiat = df_c9_base.index[idx_min]

    rezultate_ro.append({
        "Cod": cod_ro,
        "Regiune": nume_ro,
        "CelMaiApropiat": cod_apropiat,
        "Denumire": df_c9_base.loc[cod_apropiat, "Denumire"],
        "Tara": df_c9_base.loc[cod_apropiat, "Tara"],
        "Distanta": distante_ro[idx_min]
    })
df_c9 = pd.DataFrame(rezultate_ro)
# df_c9.set_index("Cod", inplace=True)
df_c9.set_index("Cod")
df_c9.to_csv("./data_out/Cerinta9.csv")

print(df_c9)

Verificare: matricea distanțelor coincide cu cdist? True
   Cod           Regiune  CelMaiApropiat            Denumire      Tara  \
0  217         Nord-Vest              20         Severozápad     Cehia   
1  218            Centru              20         Severozápad     Cehia   
2  219          Nord-Est             166  Észak-Magyarország   Ungaria   
3  220           Sud-Est             131   Panonska Hrvatska   Croatia   
4  221      Sud-Muntenia             131   Panonska Hrvatska   Croatia   
5  222   Bucureşti-Ilfov             227   Bratislavský kraj  Slovacia   
6  223  Sud-Vest Oltenia             131   Panonska Hrvatska   Croatia   
7  224              Vest             163      Közép-Dunántúl   Ungaria   

   Distanta  
0  0.722737  
1  0.541948  
2  0.803276  
3  0.882866  
4  0.895263  
5  1.965177  
6  0.460549  
7  0.502401  


**Interpretare:**
Rezultatele obținute sunt plauzibile.
 Regiunile din România se potrivesc ca nivel de dezvoltare cu regiuni vecine din Est. București-Ilfov are o distanță aproape dublă comparativ cu restul regiunilor din Romania, deoarece are un PIB și o rată a populației cu studii superioare prea mari. Astfel, singurul său rezultat apropiat din altă țară este  regiunea capitale dezvoltatate a Slovaciei.

## Cerința 10 - Grafice și concluzie

Salvați fiecare grafic în `data_out/`, la o rezoluție care permite citirea etichetelor.

**a)** *Cerinta10a.png*: scatter `PIB` vs `SperantaViata`, cu altă culoare pentru regiunile din țările care au aderat **înainte de 2004** față de cele care au aderat din 2004 încoace. Etichetați regiunile din România. Folosiți scala logaritmică pe axa PIB și explicați de ce.

**Interpretare:**

**b)** *Cerinta10b.png*: boxplot cu distribuția PIB-ului regional pe țări, cu țările ordonate după mediană. Trasați o linie orizontală la pragul de 75% din media UE (cerința 6).

**Interpretare:**

**c)** *Cerinta10c.png*: heatmap al matricei $R$ de la cerința 7, cu numele variabilelor pe axe, valorile scrise în celule și o legendă a culorilor.

**Interpretare:**

### Concluzii (aprox. 150 de cuvinte)
*Ce diferențiază regiunile bogate de celelalte?* Folosiți rezultatele de la cerințele 3-10: diferențe, coeziune, corelații, grafice.